# Ingestão de Dados

In [1]:
# Importando Bibliotecas
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when, count, avg, abs as spark_abs, column

In [2]:
# Iniciando sessão Spark
spark = SparkSession.builder \
    .appName("Studys") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession iniciada com sucesso")

SparkSession iniciada com sucesso


In [3]:
# Lendo as tabela e salvando em dfs

# Base principal (contém dados cadastrais, financeiros e TARGET)
application_train = r'DataSets/application_train.csv'
df_application_train = spark.read.csv(
    application_train,
    header=True,
    inferSchema=False,
    sep=","
)

# Histórico de crédito externo (Contém histórico em outros bancos, dividas passadas, inadimplência anterior)
bureau_csv = r"DataSets/bureau.csv"
df_bureau = spark.read.csv(
    bureau_csv,
    header=True,
    inferSchema=False,
    sep=","
)

# Histórico interno (contém pedidos anteriores, aprovações e recusas, compartamento histórico)
previous_application = r'DataSets/previous_application.csv'
df_previous_application = spark.read.csv(
    previous_application,
    header=True,
    inferSchema=False,
    sep=","
)

In [4]:
print("Application shape:", "Quantidade de dados ",df_application_train.count(), "Qtdd colunas ",len(df_application_train.columns))
print("Bureau shape:", "Quantidade de dados ", df_bureau.count(), "Qtdd colunas ", len(df_bureau.columns))
print("Previous shape:", "Quantidade de dados ", df_previous_application.count(), "Qtdd colunas ", len(df_previous_application.columns))

Application shape: Quantidade de dados  307511 Qtdd colunas  122
Bureau shape: Quantidade de dados  1716428 Qtdd colunas  17
Previous shape: Quantidade de dados  1670214 Qtdd colunas  37


# Data Understanding

In [15]:
# Selecionando colunas Relevantes
col_application_train = [
    "SK_ID_CURR"            # ID
    , "TARGET"              # variável alvo → 1 = inadimplente (default), 0 = adimplente
    , "CODE_GENDER"         # categórica nominal, gênero do cliente (M/F)
    , "FLAG_OWN_CAR"        # binária categórica, possui carro (Y/N)
    , "FLAG_OWN_REALTY"     # binária categórica, possui imóvel (Y/N)
    , "CNT_CHILDREN"        # numérica discreta (contagem), número de filhos
    , "AMT_INCOME_TOTAL"    # numérica contínua, renda total do cliente
    , "AMT_CREDIT"          # numérica contínua, valor do crédito solicitado
    , "AMT_ANNUITY"         # numérica contínua, valor da anuidade/prestação do crédito
    , "AMT_GOODS_PRICE"     # numérica contínua, preço dos bens financiados
    ,"NAME_INCOME_TYPE"     # categórica nominal, tipo de renda (trabalho, aposentado, etc.)
    , "NAME_EDUCATION_TYPE" # categórica ordinal, nível educacional
    , "NAME_FAMILY_STATUS"  # categórica nominal, estado civil
    , "NAME_HOUSING_TYPE"   # categórica nominal, tipo de moradia
    , "DAYS_BIRTH"          # numérica contínua, idade em dias (valor negativo → dias antes da aplicação)
    # Observação: geralmente convertido para anos
    , "DAYS_EMPLOYED"       # numérica contínua, dias empregado
    , "OCCUPATION_TYPE"     # categórica nominal, ocupação profissional
]

col_bureau = [
    "SK_ID_CURR"            # ID, FK
    , "SK_ID_BUREAU"        # ID, identificador único de cada registro de crédito reportado por bureaus externos (um cliente pode ter vários).
    , "CREDIT_ACTIVE"       # categórica nominal, status atual do crédito no bureau (Active, Closed, Sold, Bad debt)
    , "AMT_CREDIT_SUM"      # numérica contínua, valor total do crédito reportado pelo bureau para aquele contrato
    , "AMT_CREDIT_SUM_OVERDUE" # numérica contínua, valor em atraso (overdue). Indica inadimplência naquele crédito específico.
    , "DAYS_CREDIT"         # numérica discreta, número de dias desde a concessão do crédito até a data atual (quando negativo indicando passado).
]

col_previous_application = [
    "SK_ID_CURR"            # ID, FK
    , "SK_ID_PREV"          # ID
    , "NAME_CONTRACT_STATUS" # categórica nominal, status da aplicação anterior (Approved, Refused, Canceled, Unused offer)
    , "AMT_APPLICATION"     # numérica contínua, valor solicitado pelo cliente na aplicação anterior.
    , "AMT_CREDIT"          # numérica contínua, valor efetivamente concedido
    , "AMT_ANNUITY"         # numérica contínua, valor da parcela periódica do crédito.
    , "DAYS_DECISION"       # numérica discreta, dias desde a decisão da aplicação anterior até o momento atual (tipicamente negativo).
]

df_application_train = df_application_train.select(col_application_train)
df_bureau = df_bureau.select(col_bureau)
df_previous_application = df_previous_application.select(col_previous_application)

In [6]:
print("Application shape:", "Quantidade de dados ",df_application_train.count(), "Qtdd colunas ",len(df_application_train.columns))
print("Bureau shape:", "Quantidade de dados ", df_bureau.count(), "Qtdd colunas ", len(df_bureau.columns))
print("Previous shape:", "Quantidade de dados ", df_previous_application.count(), "Qtdd colunas ", len(df_previous_application.columns))

Application shape: Quantidade de dados  307511 Qtdd colunas  17
Bureau shape: Quantidade de dados  1716428 Qtdd colunas  6
Previous shape: Quantidade de dados  1670214 Qtdd colunas  7


In [8]:
# Identificação de linhas Nulas por Coluna - df_application_train

total_rows = df_application_train.count()

null_counts = (
    df_application_train
    .select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in df_application_train.columns
    ]).toPandas()
    .T # Transpose, transforma linhas em colunas
    .reset_index() # transforma o índice em uma coluna de cabeçalho da tb
)

null_counts.columns = ["coluna", "qtdd_nulos"] # Renomeia as colunas

null_counts["percentual (%)"] = (
    null_counts["qtdd_nulos"] / total_rows * 100
).round(2)

null_counts = (
    null_counts[null_counts["qtdd_nulos"] > 0]
    .sort_values(by="qtdd_nulos", ascending=False)
)

print("======== Colunas com valores nulos  ========")
print("========== df_application_train ===========")
print(null_counts.to_string(index=False))


======== Colunas com valores nulos  ========
========== df_application_train ===========
         coluna  qtdd_nulos  percentual (%)
OCCUPATION_TYPE       96391           31.35
AMT_GOODS_PRICE         278            0.09
    AMT_ANNUITY          12            0.00


In [9]:
# Identificação de linhas Nulas por Coluna - df_bureau

total_rows = df_bureau.count()

null_counts = (
    df_bureau
    .select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in df_bureau.columns
    ]).toPandas()
    .T # Transpose, transforma linhas em colunas
    .reset_index() # transforma o índice em uma coluna de cabeçalho da tb
)

null_counts.columns = ["coluna", "qtdd_nulos"] # Renomeia as colunas

null_counts["percentual (%)"] = (
    null_counts["qtdd_nulos"] / total_rows * 100
).round(2)

null_counts = (
    null_counts[null_counts["qtdd_nulos"] > 0]
    .sort_values(by="qtdd_nulos", ascending=False)
)

print("======== Colunas com valores nulos  ========")
print("================ df_bureau =================")
print(null_counts.to_string(index=False))


======== Colunas com valores nulos  ========
================ df_bureau =================
        coluna  qtdd_nulos  percentual (%)
AMT_CREDIT_SUM          13             0.0


In [10]:
# Identificação de linhas Nulas por Coluna - df_previous_application

total_rows = df_previous_application.count()

null_counts = (
    df_previous_application
    .select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in df_previous_application.columns
    ]).toPandas()
    .T # Transpose, transforma linhas em colunas
    .reset_index() # transforma o índice em uma coluna de cabeçalho da tb
)

null_counts.columns = ["coluna", "qtdd_nulos"] # Renomeia as colunas

null_counts["percentual (%)"] = (
    null_counts["qtdd_nulos"] / total_rows * 100
).round(2)

null_counts = (
    null_counts[null_counts["qtdd_nulos"] > 0]
    .sort_values(by="qtdd_nulos", ascending=False)
)

print("======== Colunas com valores nulos  ========")
print("========= df_previous_application ==========")
print(null_counts.to_string(index=False))


======== Colunas com valores nulos  ========
========= df_previous_application ==========
     coluna  qtdd_nulos  percentual (%)
AMT_ANNUITY      372235           22.29
 AMT_CREDIT           1            0.00


In [11]:
# Dados com incosistência (Essa análise foi realizada com base na documentaçõo do repositório com Datasets do Kaggle)
# Foi identificado uma incosistência na coluna DAYS_EMPLOYED do df  df_application_train.
# O valor 365243 aparece quando o cliente está desempregado, ele representa um valor aplicado errado na origem

total_rows = df_application_train.count()

qtd_inc = df_application_train.filter(col("DAYS_EMPLOYED") == "365243").count()
percentual = round(qtd_inc / total_rows * 100, 2)

print("=== Inconsistências em DAYS_EMPLOYED ===")
print(f"Registros com valor 365243 (representa 'desempregado' codificado incorretamente): {qtd_inc}")
print(f"Representa {percentual}% dos registros")

=== Inconsistências em DAYS_EMPLOYED ===
Registros com valor 365243 (representa 'desempregado' codificado incorretamente): 55374
Representa 18.01% dos registros


# Exploratory Data Analysis

In [17]:
# Convertendo colunas numéricas de string para double para serem analisadas

df_eda = df_application_train\
    .withColumn("AMT_INCOME_TOTAL", col("AMT_INCOME_TOTAL").cast("double"))\
    .withColumn("AMT_CREDIT", col("AMT_CREDIT").cast("double"))\
    .withColumn("AMT_ANNUITY", col("AMT_ANNUITY").cast("double"))\
    .withColumn("DAYS_BIRTH", col("DAYS_BIRTH").cast("int"))\
    .withColumn("DAYS_EMPLOYED", col("DAYS_EMPLOYED").cast("double"))\
    .withColumn("TARGET", col("TARGET").cast("int"))


In [30]:
print("====================== Análise Univariada ======================")
df_eda.select(
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY"
).describe().show()

====================== Análise Univariada ======================
+-------+-----------------+-----------------+------------------+
|summary| AMT_INCOME_TOTAL|       AMT_CREDIT|       AMT_ANNUITY|
+-------+-----------------+-----------------+------------------+
|  count|           307511|           307511|            307499|
|   mean|168797.9192969845|599025.9997057016|27108.573909183444|
| stddev| 237123.146278856|402490.7769958547|14493.737315118331|
|    min|          25650.0|          45000.0|            1615.5|
|    max|           1.17E8|        4050000.0|          258025.5|
+-------+-----------------+-----------------+------------------+



In [31]:
# Análise da variável TARGET

target_pd = df_application_train.groupBy("TARGET").count().orderBy("TARGET").toPandas()
total = target_pd["count"].sum()
target_pd["percentual (%)"] = (target_pd["count"] / total * 100).round(2)
target_pd["label"] = target_pd["TARGET"].map({"0": "Adimplente (pagou)", "1": "Inadimplente (ñ pagou)"})

print("============= Análise do TARGET =============")
print(target_pd[["label", "count", "percentual (%)"]].to_string(index=False))
print(f"\nTotal: {total} registros")

============= Análise do TARGET =============
                 label  count  percentual (%)
    Adimplente (pagou) 282686           91.93
Inadimplente (ñ pagou)  24825            8.07

Total: 307511 registros


In [32]:
# Início da análise bivariada das variáveis
# Fazendo count de Renda média, crédito médio e parcela média por caso de adimplência e inadimplência.

from pyspark.sql.functions import avg, round as spark_round

renda_credito = df_eda.groupBy("TARGET").agg(
    spark_round(avg("AMT_INCOME_TOTAL"), 2).alias("renda_media"),
    spark_round(avg("AMT_CREDIT"), 2).alias("credito_medio"),
    spark_round(avg("AMT_ANNUITY"), 2).alias("parcela_media")
).withColumn(
    "TARGET",
    when(col("TARGET") == 0, "Adimplente")
    .otherwise("Inadimplente")
).orderBy("TARGET").toPandas()


print("====== Renda, Crédito e Parcela média por TARGET ======")
print(renda_credito.to_string(index=False))


====== Renda, Crédito e Parcela média por TARGET ======
      TARGET  renda_media  credito_medio  parcela_media
  Adimplente    169077.72      602648.28       27163.62
Inadimplente    165611.76      557778.53       26481.74


In [34]:
# Análise bivariada com variaveis categóricas
# Idade média por TARGET
# DAYS_BIRTH é negativo: -9461 significa que o cliente nasceu 9461 dias antes da solicitação
idade = df_eda.withColumn("idade_anos", (spark_abs(col("DAYS_BIRTH")) / 365).cast("int")) \
    .groupBy("TARGET").agg(
        spark_round(avg("idade_anos"), 1).alias("idade_media")
    ).orderBy("TARGET").toPandas()

idade["TARGET"] = idade["TARGET"].map({0: "Adimplente", 1: "Inadimplente"})
print("=== Idade média por TARGET ===")
print(idade.to_string(index=False))

=== Idade média por TARGET ===
      TARGET  idade_media
  Adimplente         43.7
Inadimplente         40.3


In [35]:
# Taxa de inadimplência por gênero
print("=== Taxa de inadimplência por Gênero ===")
df_eda.groupBy("CODE_GENDER").agg(
    count("*").alias("total"),
    spark_round(avg(col("TARGET").cast("double")) * 100, 2).alias("taxa_inadimplencia (%)")
).orderBy("taxa_inadimplencia (%)").show()

=== Taxa de inadimplência por Gênero ===
+-----------+------+----------------------+
|CODE_GENDER| total|taxa_inadimplencia (%)|
+-----------+------+----------------------+
|        XNA|     4|                   0.0|
|          F|202448|                   7.0|
|          M|105059|                 10.14|
+-----------+------+----------------------+



In [ ]:

print("=== Taxa de inadimplência por Tipo de Renda ===")
df_eda.groupBy("NAME_INCOME_TYPE").agg(
    count("*").alias("total"),
    spark_round(avg(col("TARGET").cast("double")) * 100, 2).alias("taxa_inadimplencia (%)")
).orderBy("taxa_inadimplencia (%)", ascending=False).show()


In [33]:
# Análise de outliers das variáveis

variaveis = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY"
]

for var in variaveis:

    q1, q3 = df_eda.approxQuantile(var, [0.25, 0.75], 0)

    iqr = q3 - q1

    limite_inferior = q1 - (1.5 * iqr)
    limite_superior = q3 + (1.5 * iqr)

    outliers = df_eda.filter(
        (df_eda[var] < limite_inferior) |
        (df_eda[var] > limite_superior)
    ).count()

    print(f"\n==== {var} ====")
    print(f"Q1: {q1:.2f}")
    print(f"Q3: {q3:.2f}")
    print(f"IQR: {iqr:.2f}")
    print(f"Limite inferior: {limite_inferior:.2f}")
    print(f"Limite superior: {limite_superior:.2f}")
    print(f"Quantidade de outliers: {outliers}")


==== AMT_INCOME_TOTAL ====
Q1: 112500.00
Q3: 202500.00
IQR: 90000.00
Limite inferior: -22500.00
Limite superior: 337500.00
Quantidade de outliers: 14035

==== AMT_CREDIT ====
Q1: 270000.00
Q3: 808650.00
IQR: 538650.00
Limite inferior: -537975.00
Limite superior: 1616625.00
Quantidade de outliers: 6562

==== AMT_ANNUITY ====
Q1: 16524.00
Q3: 34596.00
IQR: 18072.00
Limite inferior: -10584.00
Limite superior: 61704.00
Quantidade de outliers: 7504


In [10]:
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.
